In [1]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [5]:
Processed_Dataset = "20261002_pallas_interpolation"

In [6]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [7]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [8]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 34688, range: 434)> Size: 120MB
array([[0.40465115, 0.40231582, 0.39984443, ...,        nan,        nan,
               nan],
       [0.4046678 , 0.40233208, 0.39986032, ...,        nan,        nan,
               nan],
       [0.40468437, 0.40234827, 0.39987613, ...,        nan,        nan,
               nan],
       ...,
       [0.50199772, 0.49921011, 0.49641896, ...,        nan,        nan,
               nan],
       [0.50199772, 0.49921011, 0.49641896, ...,        nan,        nan,
               nan],
       [0.50199772, 0.49921011, 0.49641896, ...,        nan,        nan,
               nan]], shape=(34688, 434))
Coordinates:
  * Time     (Time) datetime64[ns] 278kB 2026-10-02T00:46:20.572915077 ... 20...
  * range    (range) float64 3kB 223.6 238.5 253.4 ... 1.094e+04 1.097e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 34688, range

In [9]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")